# Support Vector Machine - Heart Disease

Explores a tuned SVM classifier on the cleaned splits.

**Method.** Hyperparameters are tuned with stratified 5-fold `GridSearchCV` on the
**training** set; the tuned model is evaluated on the **validation** set. The
**test** set is deliberately left untouched here - it is reserved for the final
cross-model comparison notebook, so the hold-out is spent only once.

SVM needs scaled features because margin geometry depends on feature scale.
Scaling is applied to the continuous columns only, inside the pipeline, via
`dataset.build_scaler`.

In [ ]:
import sys
from pathlib import Path

# Make the scripts/ helpers importable from notebooks/.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score, f1_score, accuracy_score,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay,
)

from dataset import load_splits, get_xy
from train_models import build_pipeline, MODELS, CV

MODEL_NAME = "svm"

## Load data

In [ ]:
train, val, test = load_splits()
X_train, y_train = get_xy(train)
X_val, y_val = get_xy(val)

print(f"train {X_train.shape}  val {X_val.shape}  (test held back)")
print(f"train disease rate {y_train.mean():.3f}  val {y_val.mean():.3f}")

## Tune hyperparameters (train, 5-fold CV)

The estimator and grid come straight from the `MODELS` registry in
`train_models.py`, so the notebook and the batch script stay in sync.

In [ ]:
cfg = MODELS[MODEL_NAME]
pipe = build_pipeline(cfg["estimator"], cfg["needs_scaling"], list(X_train.columns))
search = GridSearchCV(pipe, cfg["grid"], cv=CV, scoring="roc_auc", n_jobs=-1)
search.fit(X_train, y_train)

best = search.best_estimator_
print("best params:", search.best_params_)
print(f"cv roc-auc: {search.best_score_:.3f}")

## Validation metrics

One caveat specific to SVC: with `probability=True` the probabilities come from
Platt scaling (a sigmoid fit via internal cross-validation), so `predict_proba`
can occasionally disagree with the hard `predict` labels near the boundary.
ROC-AUC uses the probabilities; the threshold metrics below use `predict`.

In [ ]:
proba = best.predict_proba(X_val)[:, 1]
pred = best.predict(X_val)

metrics = {
    "roc_auc": roc_auc_score(y_val, proba),
    "recall": recall_score(y_val, pred),
    "precision": precision_score(y_val, pred),
    "f1": f1_score(y_val, pred),
    "accuracy": accuracy_score(y_val, pred),
}
pd.Series(metrics, name=MODEL_NAME).round(3)

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix(y_val, pred), display_labels=["no disease", "disease"]
).plot(cmap="Blues", colorbar=False)
plt.title(f"{MODEL_NAME} - validation confusion matrix")
plt.show()

## Hyperparameter comparison

Compare the cross-validated ROC-AUC for each SVM configuration. `C` controls
regularization strength, `kernel` chooses the decision boundary shape, and
`gamma` controls RBF locality. `gamma` only applies to the RBF kernel, so the
linear-kernel rows are deduplicated below.

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
param_cols = ["param_model__C", "param_model__kernel", "param_model__gamma"]

svm_grid = cv_results[param_cols + ["mean_test_score", "std_test_score"]].copy()
svm_grid = svm_grid.rename(columns={
    "param_model__C": "C",
    "param_model__kernel": "kernel",
    "param_model__gamma": "gamma",
    "mean_test_score": "cv_roc_auc",
    "std_test_score": "cv_std",
})

# gamma is ignored by the linear kernel, so linear rows differing only in
# gamma are the same model - blank it out and drop the duplicates.
svm_grid.loc[svm_grid["kernel"] == "linear", "gamma"] = "-"
svm_grid = svm_grid.drop_duplicates(subset=["C", "kernel", "gamma"])

svm_grid.sort_values("cv_roc_auc", ascending=False).round(3)

In [ ]:
plot_grid = svm_grid.sort_values("cv_roc_auc", ascending=True).copy()
plot_grid["label"] = plot_grid.apply(
    lambda row: f"C={row['C']}, {row['kernel']}"
    + (f", gamma={row['gamma']}" if row["kernel"] == "rbf" else ""),
    axis=1,
)

plt.figure(figsize=(8, 5))
plt.barh(plot_grid["label"], plot_grid["cv_roc_auc"], xerr=plot_grid["cv_std"], color="steelblue")
plt.xlabel("CV ROC-AUC")
plt.title("SVM - hyperparameter CV comparison")
plt.xlim(max(0.0, plot_grid["cv_roc_auc"].min() - 0.03), min(1.0, plot_grid["cv_roc_auc"].max() + 0.03))
plt.show()

## Support vectors

The number of support vectors gives a quick sense of how complex the fitted
boundary is. A model relying on many training points may be fitting a more
flexible boundary than a model with fewer support vectors.

In [ ]:
support_counts = pd.Series(
    best.named_steps["model"].n_support_, index=["no disease", "disease"], name="support_vectors"
)
support_counts

In [ ]:
support_counts.plot(kind="bar", color=["tab:blue", "tab:red"], rot=0)
plt.ylabel("count")
plt.title("SVM - support vectors by class")
plt.show()

## ROC curve (validation)

In [ ]:
fpr, tpr, _ = roc_curve(y_val, proba)
plt.plot(fpr, tpr, label=f"{MODEL_NAME} (AUC={metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], ls="--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("SVM - validation ROC curve")
plt.legend()
plt.show()

## Notes

- Test set intentionally not touched - see the comparison notebook for the
  single final test evaluation of the selected model.
- This notebook is SVM specific. To explore another model, start from the
  shared template and swap in a model-appropriate diagnostic.